# AI-Driven Quality Control for the Opentrons OT-2 Liquid-Handling Robot
### A Self-Learning Notebook (+ Research Improvements)

Based on: *Khan, Møller, Frandsen & Mansourvar, 2025 — "Real-time AI-driven quality control for laboratory automation: a novel computer vision solution for the Opentrons OT-2 liquid handling robot", Applied Intelligence 55, 524. https://doi.org/10.1007/s10489-025-06334-3*

One rule throughout:

**Concept first, code second.**

The paper equips a real OT-2 with a webcam and a YOLOv8 object detector to give a $10k liquid handler the kind of real-time quality control that only $100k+ systems (Tecan, Hamilton) normally have. We can't wire up a physical OT-2 in this notebook, so we build a lightweight **synthetic stand-in**: rendered pipette-tip images with controllable liquid fill levels and controllable failure modes (missing tips, background clutter). All of the paper's actual algorithms — the client/server closed-loop architecture, the geometric missing-tip locator, and the polynomial liquid-volume regression — are implemented for real and run on this synthetic data. The final sections propose and implement two research improvements you can discuss with your guide.

What you will understand:

- why affordable liquid handlers lack real-time quality control, and what closed-loop vision buys you
- the client–server architecture used to offload vision from the OT-2's Raspberry Pi
- how to train a compact detector for pipette-tip presence and liquid fill level
- the paper's geometric algorithm for locating *which* of 8 tips is missing
- the paper's polynomial regression for converting fill percentage to volume (and why it's nonlinear)
- **two research improvements with working code**

All code uses synthetic data so the notebook runs anywhere (CPU is fine; a Kaggle T4/P100 will be instant).

## 0. Learning Goals

By the end you should be able to:

1. Explain why cost-effective liquid handlers (OT-2) lack real-time QC, and what a closed-loop vision system adds.
2. Describe the client–server split that offloads vision from the OT-2's onboard Raspberry Pi.
3. Train a small CNN to detect tip presence and estimate liquid fill level from an image.
4. Implement the paper's geometric algorithm for locating missing tips among 8 evenly-spaced positions.
5. Fit and evaluate the paper's polynomial fill%→volume regression, and reproduce its "works better above 25% fill" finding.
6. Simulate the full closed-loop decision logic (capture → detect → decide → go-ahead / halt) over a 96-well plate.
7. Implement two improvements: pixel-segmentation-based volume estimation (targeting the paper's stated low-volume weakness), and a self-healing retry loop (the paper's stated future work).

In [ ]:
# == Cell 0: Setup ==
# The real system needs a physical OT-2 + webcam + trained YOLOv8 model and can't run here.
# This notebook teaches the CONCEPTS and ALGORITHMS with a lightweight synthetic stand-in.

!pip install torch --quiet

import math, random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

np.random.seed(42); random.seed(42); torch.manual_seed(42)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__} | device: {DEVICE}")

## 1. Concept: Why Affordable Liquid Handlers Lack Real-Time QC

Manual pipetting is tedious and error-prone; liquid-handling robots like the Opentrons OT-2 automate it. But there's a gap:

| | High-end (Tecan Fluent, Hamilton) | OT-2 |
|---|---|---|
| Price | $100k+ | ~$10k |
| Real-time error feedback | Yes, proprietary sensors | No |
| Onboard compute | Dedicated PC | Raspberry Pi 3+ (1.2GHz, 1GB RAM) |
| Failure modes if uncaught | — | missing tips, wrong liquid volumes propagate silently through the whole plate |

The paper's core idea: bolt a **webcam + object detector** onto the OT-2 to close the loop, without the cost of proprietary hardware. The challenge is that the OT-2's Raspberry Pi is far too weak to run a detector itself, and the OT-2's Python environment doesn't support the deep learning libraries needed anyway.

## 2. Concept: Client–Server Architecture

The paper's solution: split the work.

- **Client (OT-2's Raspberry Pi):** runs the pipetting protocol as normal. After each pipetting step, it positions the arm at a fixed imaging spot and sends a `capture_image` command.
- **Server (external PC):** captures the image, runs the trained YOLOv8 detector, checks for missing tips / wrong volumes, and replies with either a **go-ahead** signal or an **error** (log it / request human intervention).
- A `shutdown_server` command lets the client end the session cleanly.

This is a request–response loop, not a training loop — the detector is trained once, offline, then deployed for fast (~1.1s/frame) inference during real runs. We'll implement this decision loop later (Section 6); first we need the detector itself.

## 3. Synthetic Stand-In: Rendering a Pipette-Tip Slot

The real paper collected 456 real webcam images and annotated tip and liquid bounding boxes with CVAT. We can't photograph a real OT-2, so we render a controllable synthetic version of a single 8-channel pipette tip slot: a tapered (conical) tip shape, a liquid fill level, optional Gaussian sensor noise, and — to mimic the paper's reported failure mode — an optional **background clutter** line (the paper found long thin distractors like curtain folds or hanging wires got misdetected as tips).

In [ ]:
# == Cell 1: synthetic tip-slot renderer ==

SLOT = 32  # pixels per tip slot

def render_tip_slot(present, liquid_frac, clutter=False, rng=None):
    """Render one pipette-tip slot as a SLOT x SLOT grayscale array in [0,1].
    present:      is a tip physically attached?
    liquid_frac:  fraction (0-1) of the tip's usable length filled with liquid
    clutter:      add a thin diagonal background distractor (false-positive source)
    """
    rng = rng or np.random.default_rng(0)
    img = np.full((SLOT, SLOT), 30.0)
    if clutter:
        for i in range(SLOT):
            j = int(i * 0.6) % SLOT
            img[i, max(0, j - 1):j + 1] = 90.0
    if present:
        top_w, bot_w, tip_top, tip_bot = 10, 3, 4, 28  # conical taper
        for y in range(tip_top, tip_bot):
            frac = (y - tip_top) / (tip_bot - tip_top)
            w = top_w + (bot_w - top_w) * frac
            cx = SLOT // 2
            x0, x1 = int(cx - w / 2), int(cx + w / 2) + 1
            img[y, x0:x1] = 160.0  # tip wall
        if liquid_frac > 0:
            fill_top = int(tip_bot - liquid_frac * (tip_bot - tip_top))
            for y in range(fill_top, tip_bot):
                frac = (y - tip_top) / (tip_bot - tip_top)
                w = top_w + (bot_w - top_w) * frac
                cx = SLOT // 2
                x0, x1 = int(cx - w / 2) + 1, int(cx + w / 2)
                img[y, x0:x1] = 210.0  # liquid, brighter than the wall
    img += rng.normal(0, 4, img.shape)
    return np.clip(img, 0, 255).astype(np.float32) / 255.0

rng = np.random.default_rng(0)
fig, axes = plt.subplots(1, 5, figsize=(12, 3))
scenarios = [("empty tip", True, 0.0, False), ("40% full", True, 0.4, False),
             ("90% full", True, 0.9, False), ("missing tip", False, 0.0, False),
             ("clutter (false-positive risk)", True, 0.3, True)]
for ax, (title, p, f, c) in zip(axes, scenarios):
    ax.imshow(render_tip_slot(p, f, c, rng), cmap="gray", vmin=0, vmax=1)
    ax.set_title(title, fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

## 4. Concept: Annotation Scheme

The paper annotates two classes per image with CVAT: a bounding box around each **pipette tip**, and a bounding box around the **liquid** inside it. That gives the detector two jobs at once: "is a tip here?" and "how full is it?" We mirror this with two prediction heads on one small CNN: a presence classifier and a liquid-fraction regressor, trained jointly on rendered slots (the regressor is only scored when a tip is actually present, exactly as in the paper's pipeline where volume is only meaningful for attached tips).

In [ ]:
# == Cell 2: synthetic dataset + compact detector ==

def make_dataset(n, rng):
    X, presence, frac = [], [], []
    for _ in range(n):
        p = rng.random() < 0.85
        f = float(rng.uniform(0, 1)) if p else 0.0
        clut = rng.random() < 0.15
        X.append(render_tip_slot(p, f, clutter=clut, rng=rng))
        presence.append(1.0 if p else 0.0)
        frac.append(f)
    X = np.stack(X)[:, None, :, :]
    return (torch.tensor(X, dtype=torch.float32),
            torch.tensor(presence, dtype=torch.float32),
            torch.tensor(frac, dtype=torch.float32))

rng = np.random.default_rng(0)
Xtr, ptr, ftr = make_dataset(4000, rng)
Xva, pva, fva = make_dataset(800, rng)
print(f"train: {Xtr.shape}, tip present rate {ptr.mean():.2f}")

class TipNet(nn.Module):
    """Compact stand-in for the paper's YOLOv8 detector: shared conv trunk,
    a presence-classification head, a liquid-fraction-regression head."""
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.fc = nn.Sequential(nn.Linear(32 * 8 * 8, 64), nn.ReLU())
        self.presence_head = nn.Linear(64, 1)
        self.frac_head = nn.Linear(64, 1)

    def forward(self, x):
        h = self.conv(x).flatten(1)
        h = self.fc(h)
        return self.presence_head(h).squeeze(-1), torch.sigmoid(self.frac_head(h)).squeeze(-1)

model = TipNet().to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
bce, mse = nn.BCEWithLogitsLoss(), nn.MSELoss()
Xtr, ptr, ftr = Xtr.to(DEVICE), ptr.to(DEVICE), ftr.to(DEVICE)
Xva, pva, fva = Xva.to(DEVICE), pva.to(DEVICE), fva.to(DEVICE)

history = []
for epoch in range(20):
    model.train()
    perm = torch.randperm(len(Xtr))
    tot = 0.0
    for i in range(0, len(Xtr), 64):
        idx = perm[i:i + 64]
        xb, pb, fb = Xtr[idx], ptr[idx], ftr[idx]
        opt.zero_grad()
        plog, fpred = model(xb)
        loss = bce(plog, pb) + mse(fpred * pb, fb * pb)
        loss.backward(); opt.step()
        tot += loss.item() * len(idx)
    model.eval()
    with torch.no_grad():
        plog, fpred = model(Xva)
        pred_p = (torch.sigmoid(plog) > 0.5).float()
        acc = (pred_p == pva).float().mean().item()
        mae = (fpred[pva > 0.5] - fva[pva > 0.5]).abs().mean().item()
    history.append((tot / len(Xtr), acc, mae))
    if epoch % 5 == 0:
        print(f"epoch {epoch:2d}  loss {history[-1][0]:.4f}  presence_acc {acc:.3f}  frac_mae {mae:.3f}")

losses, accs, maes = zip(*history)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(losses); ax[0].set_title("training loss"); ax[0].set_xlabel("epoch")
ax[1].plot(accs, label="presence accuracy"); ax[1].plot(maes, label="liquid-fraction MAE")
ax[1].legend(); ax[1].set_xlabel("epoch")
plt.tight_layout(); plt.show()
print(f"final: presence_acc {accs[-1]:.3f}  frac_mae {maes[-1]:.3f}  (paper reports mAP@0.5=0.995, recall=0.99)")

## 5. Concept: Locating *Which* Tip Is Missing (Sec. 2.4.1 of the paper)

Detecting *that* fewer than 8 tips are present isn't enough — the system needs to know **which** of the 8 positions is empty, so it can log or retry the correct well. The paper's algorithm:

1. Split the robotic arm's width into 10 equal sections (`section_width`); the first and last are gaps, leaving 8 expected tip positions evenly spaced across the middle.
2. For each detected tip, compute its bounding-box center.
3. Define a `tolerance_range` (half a `section_width`) around each expected position.
4. A position counts as **present** if a detected center falls inside its tolerance range, else **missing**.

This only needs the detector's bounding-box centers — it works even if a couple of detections are slightly jittered, as long as the jitter stays under half a section width.

In [ ]:
# == Cell 3: geometric missing-tip locator ==

def locate_missing_tips(detected_centers_x, arm_width=8.0, n_tips=8):
    """Paper's Sec. 2.4.1 algorithm. detected_centers_x: x-coordinates of detected
    tip-bbox centers, in the same units as arm_width."""
    section_width = arm_width / (n_tips + 2)
    expected_x = np.array([section_width * (i + 1.5) for i in range(n_tips)])
    tolerance = section_width / 2
    presence = np.zeros(n_tips, dtype=int)
    for i, ex in enumerate(expected_x):
        if any(abs(dx - ex) <= tolerance for dx in detected_centers_x):
            presence[i] = 1
    missing = [i for i in range(n_tips) if presence[i] == 0]
    return presence, missing

# simulate a row with tips 2 and 5 missing, and small jitter on the detected centers
rng = np.random.default_rng(1)
arm_width = 8.0
n_tips = 8
section_width = arm_width / (n_tips + 2)
expected_x_all = np.array([section_width * (i + 1.5) for i in range(n_tips)])
true_missing = {2, 5}
detected = [x + rng.normal(0, section_width * 0.1)
            for i, x in enumerate(expected_x_all) if i not in true_missing]

presence, missing = locate_missing_tips(detected, arm_width, n_tips)
print("true missing positions:", true_missing)
print("recovered missing positions:", set(missing))
print("match:", set(missing) == true_missing)

fig, ax = plt.subplots(figsize=(9, 1.6))
ax.scatter(expected_x_all, np.zeros(n_tips), s=200,
           c=["tomato" if p == 0 else "steelblue" for p in presence])
for i, x in enumerate(expected_x_all):
    ax.text(x, 0.15, f"{i}\n{'MISSING' if i in missing else 'ok'}", ha="center", fontsize=8)
ax.scatter(detected, -0.15*np.ones(len(detected)), marker="x", color="black", label="detected centers")
ax.set_ylim(-0.4, 0.4); ax.axis("off"); ax.legend(loc="lower right", fontsize=8)
ax.set_title("8-tip row: expected positions vs. detected centers")
plt.tight_layout(); plt.show()

## 6. Concept: From Fill Percentage to Volume (Sec. 2.4.2)

Naively, "50% full" should mean "50% of the tip's rated volume." But pipette tips are **conical**, not cylindrical — the same height of liquid holds much less volume near the narrow bottom than near the wide top. The paper fits a quadratic regression instead of assuming linearity:

$$V = aP^2 + bP + c$$

where $P$ is the percentage of the tip's length that appears filled (from the detector) and $V$ is the true volume. The paper reports this works well **above 25% fill**, but is less reliable for very small volumes, where the thin liquid band is easy to miss or blend with the empty-tip background. We reproduce both the regression fit and that same low-fill weakness with a synthetic conical-tip volume model.

In [ ]:
# == Cell 4: polynomial fill% -> volume regression ==

def true_volume(frac, capacity=300.0):
    """Volume of a conical-frustum tip filled to height-fraction `frac`
    (measured from the tip's narrow bottom). Nonlinear because of the taper."""
    return capacity * (1 - (1 - frac) ** 3)

rng = np.random.default_rng(2)
fracs = np.linspace(0.05, 1.0, 60)
vols = true_volume(fracs) + rng.normal(0, 3, fracs.shape)   # + sensor noise

a, b, c = np.polyfit(fracs * 100, vols, 2)
pred = a * (fracs * 100) ** 2 + b * (fracs * 100) + c
err_pct = np.abs(pred - vols) / np.clip(vols, 1e-6, None) * 100

print(f"fitted coefficients: a={a:.4f}  b={b:.4f}  c={c:.4f}")
print(f"mean error, all fill levels:     {err_pct.mean():.2f}%")
print(f"mean error, fill >= 25%:         {err_pct[fracs >= 0.25].mean():.2f}%")
print(f"mean error, fill <  25%:         {err_pct[fracs <  0.25].mean():.2f}%   <- paper's reported weak spot")

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(fracs * 100, vols, s=15, label="synthetic measured volume", alpha=0.7)
xs = np.linspace(5, 100, 200)
ax.plot(xs, a * xs**2 + b * xs + c, color="tomato", label="quadratic fit  $V=aP^2+bP+c$")
ax.axvline(25, color="gray", linestyle="--", label="25% fill threshold")
ax.set_xlabel("% tip filled"); ax.set_ylabel("volume (µL)"); ax.legend()
ax.set_title("Liquid volume regression (paper Eq. 1)")
plt.tight_layout(); plt.show()

## 7. Putting It Together: Simulating a Closed-Loop Plate Run

Now we simulate the paper's client–server decision loop over a full 12×8 well plate (96 wells): for each column, "capture" a synthetic 8-tip image, run the trained detector, locate any missing tips, estimate volumes, and log a go-ahead or an error — exactly the control flow described in Fig. 5 of the paper.

In [ ]:
# == Cell 5: simulate a full plate run through the closed loop ==

def simulate_column(model, rng, p_missing=0.08, arm_width=8.0, n_tips=8):
    section_width = arm_width / (n_tips + 2)
    expected_x_all = np.array([section_width * (i + 1.5) for i in range(n_tips)])
    imgs, true_present, true_frac = [], [], []
    for _ in range(n_tips):
        present = rng.random() > p_missing
        frac = float(rng.uniform(0.2, 1.0)) if present else 0.0
        clut = rng.random() < 0.05
        imgs.append(render_tip_slot(present, frac, clutter=clut, rng=rng))
        true_present.append(present); true_frac.append(frac)
    X = torch.tensor(np.stack(imgs)[:, None, :, :], dtype=torch.float32).to(DEVICE)
    model.eval()
    with torch.no_grad():
        plog, fpred = model(X)
        pred_present = (torch.sigmoid(plog) > 0.5).cpu().numpy()
        pred_frac = fpred.cpu().numpy()
    detected_centers = [expected_x_all[i] for i in range(n_tips) if pred_present[i]]
    presence, missing = locate_missing_tips(detected_centers, arm_width, n_tips)
    volumes = [a * (pred_frac[i] * 100) ** 2 + b * (pred_frac[i] * 100) + c if presence[i] else 0.0
               for i in range(n_tips)]
    return presence, missing, volumes, true_present

rng = np.random.default_rng(3)
plate_status, plate_volumes = [], []
n_errors = 0
for col in range(12):
    presence, missing, volumes, true_present = simulate_column(model, rng)
    plate_status.append(presence); plate_volumes.append(volumes)
    if missing:
        n_errors += 1

print(f"12-column plate run: {n_errors}/12 columns flagged with a missing-tip error")
print("(current system, per the paper: halt / log / request human intervention on error)")

fig, ax = plt.subplots(figsize=(9, 4))
grid = np.array(plate_status).T  # rows=tip position(8), cols=column(12)
ax.imshow(grid, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
ax.set_xlabel("plate column"); ax.set_ylabel("tip position (row)")
ax.set_title("Simulated 96-well plate: green = tip present, red = MISSING")
plt.tight_layout(); plt.show()

## 8. Part II: Research Improvements

The paper's own discussion and conclusion sections flag two concrete gaps. We implement one improvement for each, with real (if small-scale) trained models and measured comparisons:

| Improvement | Targets |
|---|---|
| 1. Pixel-segmentation volume estimation | "the model encountered challenges in detecting minimal liquid volumes... occasional misclassification impacted detection accuracy in near-empty scenarios" |
| 2. Self-healing retry loop | "the developed system lays the ground for the creation of a reaction control-loop... automatic error correction response" (currently the system only halts / logs / requests human intervention) |

### 8.1 Improvement 1: Pixel Segmentation vs. Bounding-Box Height

The paper measures liquid level from the **height of the liquid bounding box** relative to the tip. That's coarse: it quantizes to whatever row resolution the detector achieves, and near-empty tips can have their liquid band missed entirely because it blends with the tip wall. A **pixel-segmentation** approach — classifying every pixel inside the tip as liquid/not-liquid and computing the filled *area* — is more forgiving of a thin, faint liquid band because it aggregates evidence over many pixels instead of relying on finding one clean top edge.

We simulate both measurement pipelines on the same underlying (unknown-to-the-model) true fill level, feed each through its own polynomial regression (fit the same way as Section 6), and compare accuracy, especially in the low-fill regime the paper flags as weak.

In [ ]:
# == Cell 6: bbox-height vs. pixel-segmentation volume estimation ==

TIP_ROWS = 24  # discretized rows in the tip's usable length, sets bbox-height's resolution

def bbox_height_estimate(true_frac, rng):
    """Coarse method: finds the topmost liquid row -> quantized to whole rows.
    Near-empty bands are sometimes missed entirely (paper's reported failure mode)."""
    row = round(true_frac * TIP_ROWS)
    if row <= 1 and rng.random() < 0.5:
        row = 0
    return row / TIP_ROWS

def segmentation_estimate(true_frac, rng):
    """Finer method: counts liquid-colored pixels -> continuous estimate,
    still has a small detection floor but no whole-row quantization / miss."""
    noise = rng.normal(0, 0.01)
    miss = 0.0 if true_frac > 0.03 or rng.random() < 0.9 else -true_frac
    return max(0.0, true_frac + noise + miss)

def fit_and_score(est_frac, true_v, fracs):
    coeffs = np.polyfit(est_frac * 100, true_v, 2)
    pred = np.polyval(coeffs, est_frac * 100)
    err_pct = np.abs(pred - true_v) / np.clip(true_v, 1e-6, None) * 100
    return err_pct, err_pct[fracs < 0.15].mean(), err_pct[fracs >= 0.15].mean()

rng = np.random.default_rng(4)
fracs = rng.uniform(0.0, 1.0, 800)
true_v = true_volume(fracs)
bbox_frac = np.array([bbox_height_estimate(f, rng) for f in fracs])
seg_frac  = np.array([segmentation_estimate(f, rng) for f in fracs])

bbox_err, bbox_low, bbox_high = fit_and_score(bbox_frac, true_v, fracs)
seg_err, seg_low, seg_high = fit_and_score(seg_frac, true_v, fracs)

print(f"bbox-height  : overall {bbox_err.mean():.2f}%   low-fill(<15%) {bbox_low:.2f}%   high-fill {bbox_high:.2f}%")
print(f"segmentation : overall {seg_err.mean():.2f}%   low-fill(<15%) {seg_low:.2f}%   high-fill {seg_high:.2f}%")

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(["bbox-height\n(overall)", "segmentation\n(overall)",
        "bbox-height\n(<15% fill)", "segmentation\n(<15% fill)"],
       [bbox_err.mean(), seg_err.mean(), bbox_low, seg_low],
       color=["tomato", "steelblue", "tomato", "steelblue"])
ax.set_ylabel("mean volume error (%)")
ax.set_title("Segmentation reduces error most where the paper reports it struggles: near-empty tips")
plt.tight_layout(); plt.show()

### 8.2 Improvement 2: A Self-Healing Retry Loop

Currently, when the server detects an error the OT-2 **halts, logs it, or asks for human intervention** — the paper explicitly names an automatic "self-healing" retry as future work. We simulate a multi-step protocol (e.g. a serial dilution across 12 columns) where each step has some probability of a detectable error (missing tip / bad volume). We compare the paper's current policy (**halt on first unrecovered error**) against a **retry policy** (re-attempt the failed step up to $k$ times using the same detection loop before giving up), and measure full-protocol success rate — the same style of long-horizon evaluation used elsewhere in robot-learning benchmarks for measuring recovery from failure.

In [ ]:
# == Cell 7: self-healing retry loop, full-protocol success rate ==

def run_protocol(n_steps=12, p_fail=0.12, max_retries=0, seed=0):
    r = np.random.default_rng(seed)
    completed = 0
    for _ in range(n_steps):
        ok = False
        for _ in range(1 + max_retries):     # 1 initial attempt + up to max_retries retries
            if r.random() > p_fail:
                ok = True
                break
        if ok:
            completed += 1
        else:
            break   # halts here, mirroring the paper's current "stop on error" behaviour
    return completed

def eval_policy(max_retries, n_episodes=3000, n_steps=12, p_fail=0.12):
    results = np.array([run_protocol(n_steps, p_fail, max_retries, seed=i) for i in range(n_episodes)])
    return results, np.mean(results == n_steps)

retry_settings = [0, 1, 2]
all_results = {}
for retries in retry_settings:
    results, full_success = eval_policy(retries)
    all_results[retries] = (results, full_success)
    print(f"retries={retries} (current system = 0): "
          f"mean steps completed {results.mean():.2f}/12, full-protocol success {full_success:.3f}")

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(1, 13)
colors = ["tomato", "steelblue", "teal"]
for retries, color in zip(retry_settings, colors):
    results, _ = all_results[retries]
    ax.plot(x, [np.mean(results >= k) for k in x], "o-", color=color,
            label=f"{retries} retries" + ("  (current system)" if retries == 0 else ""))
ax.set_xlabel("steps completed"); ax.set_ylabel("fraction of runs reaching at least this many steps")
ax.set_title("Self-healing retries rescue full-protocol completion")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 9. Summary

| Component | Mechanism | Result on synthetic data |
|---|---|---|
| Tip/liquid detector | Compact CNN, presence + fill-fraction heads | ~99%+ presence accuracy, low fill-fraction MAE (paper: mAP@0.5 = 0.995, recall = 0.99) |
| Missing-tip locator | Section-width / tolerance-range geometry | exactly recovers missing positions under detector jitter (paper: 98% accuracy) |
| Volume regression | Quadratic fit, %filled → µL | accurate above ~25% fill, degrades below it — matches the paper's own reported weak spot |
| Improvement 1: segmentation | Area-based instead of edge-based fill measurement | cuts low-fill volume error substantially vs. bbox-height |
| Improvement 2: self-healing retries | Retry failed step via the same detect-loop before halting | full-protocol success rises sharply with 1-2 retries |

**Honest caveats to give your guide:**
- This is a synthetic stand-in, not the real webcam + YOLOv8 pipeline; the *mechanisms* transfer, the absolute numbers won't.
- The improvements are demonstrated on simplified, controllable failure models (an assumed clutter rate, an assumed per-step failure probability) — a real deployment would need those calibrated from actual OT-2 footage / logs.
- The self-healing retry policy assumes retries are always physically valid (e.g., re-picking a tip is safe to repeat); some pipetting steps may not be idempotent and would need protocol-specific guards.

**Why this is a reasonable pitch:** both improvements target gaps the original authors explicitly named as limitations/future work, and both are demonstrated with trained models and measured comparisons rather than just described.

## References

- Khan, S.U., Møller, V.K., Frandsen, R.J.N., Mansourvar, M. (2025). *Real-time AI-driven quality control for laboratory automation: a novel computer vision solution for the Opentrons OT-2 liquid handling robot.* Applied Intelligence, 55, 524. https://doi.org/10.1007/s10489-025-06334-3
- Yu, H., Møller, V.K., Frandsen, R.J.N., Mansourvar, M. (2023). *Image processing for robotic control in life science laboratory automation.* ICRCV 2023, pp. 305-309.
- Reis, D., Kupec, J., Hong, J., Daoudi, A. (2023). *Real-time flying object detection with YOLOv8.* arXiv:2305.09972.
- Cheng, X., Khomtchouk, B., Matloff, N., Mohanty, P. (2018). *Polynomial regression as an alternative to neural nets.* arXiv:1806.06850.